In [11]:
from pathlib import Path

from Bio import Align, SeqIO
from Bio.Align import substitution_matrices

DATA_DIR = Path.cwd().parent / "data"

HUMAN_SEQ_PATH = DATA_DIR / "tfr1_human.fasta"
MOUSE_SEQ_PATH = DATA_DIR / "tfr1_mouse.fasta"

In [12]:
def read_sequence(path: Path) -> tuple[str, str]:
    """Return (id, sequence) from a FASTA file or a plain-text sequence file."""
    text = path.read_text()
    if text.lstrip().startswith(">"):
        record = next(SeqIO.parse(path, "fasta"))
        return record.id, str(record.seq).upper()
    return path.stem, "".join(text.split()).upper()


human_id, human_seq = read_sequence(SEQ_A_PATH)
mouse_id, mouse_seq = read_sequence(SEQ_B_PATH)

print(f"{human_id}: {len(human_seq)} aa")
print(f"{mouse_id}: {len(mouse_seq)} aa")

Human: 640 aa
Mouse: 641 aa


In [13]:
aligner = Align.PairwiseAligner()
aligner.mode = "global"
aligner.substitution_matrix = substitution_matrices.load("BLOSUM62")
aligner.open_gap_score = -11
aligner.extend_gap_score = -1

alignments = aligner.align(human_seq, mouse_seq)
alignment = alignments[0]

print(f"score:      {alignment.score:.1f}")
print(f"alignments: {len(alignments)}")

score:      2648.0
alignments: 3


In [14]:
def show_alignment(alignment, name_a: str, name_b: str, width: int = 80) -> None:
    """Print the alignment in blocks with a match/mismatch track and residue numbering."""
    a_aln, b_aln = alignment[0], alignment[1]
    label = max(len(name_a), len(name_b), 5)
    pos_a = pos_b = 0

    for start in range(0, len(a_aln), width):
        a_chunk = a_aln[start : start + width]
        b_chunk = b_aln[start : start + width]
        track = "".join(
            "|" if x == y else (" " if x == "-" or y == "-" else ".")
            for x, y in zip(a_chunk, b_chunk, strict=True)
        )
        start_a, start_b = pos_a + 1, pos_b + 1
        pos_a += sum(c != "-" for c in a_chunk)
        pos_b += sum(c != "-" for c in b_chunk)

        print(f"{name_a:<{label}} {start_a:>5} {a_chunk} {pos_a}")
        print(f"{'':<{label}} {'':>5} {track}")
        print(f"{name_b:<{label}} {start_b:>5} {b_chunk} {pos_b}")
        print()


show_alignment(alignment, human_id, mouse_id)

Human     1 RLYWDDLKRKLSEKLDSTDFTGTIKLLNENSYVPREAGSQKDENLALYVENQFREFKLSKVWRDQHFVKIQVKDS-AQNS 79
            ||||.|||..|||||.|..|..|||.|..|.|.||||||||||.||.|.||||.|||.||||||.|.||||||.| .||.
Mouse     1 RLYWADLKTLLSEKLNSIEFADTIKQLSQNTYTPREAGSQKDESLAYYIENQFHEFKFSKVWRDEHYVKIQVKSSIGQNM 80

Human    80 VIIVDKNGRLVYLVENPGGYVAYSKAATVTGKLVHANFGTKKDFEDLYTPVNGSIVIVRAGKITFAEKVANAESLNAIGV 159
            |.||..||.| ..||.|.||||.||...|.|||||||||||||||.|...||||.||||||.||||||||||.|.|||||
Mouse    81 VTIVQSNGNL-DPVESPEGYVAFSKPTEVSGKLVHANFGTKKDFEELSYSVNGSLVIVRAGEITFAEKVANAQSFNAIGV 159

Human   160 LIYMDQTKFPIVNAELSFFGHAHLGTGDPYTPGFPSFNHTQFPPSRSSGLPNIPVQTISRAAAEKLFGNMEGDCPSDWKT 239
            |||||..|||.|.|.|..|||||||||||||||||||||||||||.||||||||||||||||||||||.|||.||..|..
Mouse   160 LIYMDKNKFPVVEADLALFGHAHLGTGDPYTPGFPSFNHTQFPPSQSSGLPNIPVQTISRAAAEKLFGKMEGSCPARWNI 239

Human   240 DSTCRMVTSESKNVKLTVSNVLKEIKILNIFGVIKGFVEPDHYVVVGAQRDAWGPG-AAKSGVGTALLLKLAQMFSDMVL 318
            ||.|....|...||||.|.|||||..|||

In [ ]:
# https://proteinbase.com/competitions/anthropic-adaptyv-2026/challenges/tfr1
# "human D204, R208, V210, Y211, N348 and K371 become Q207, N211, D213, P214, K350 and Q373, and L212 has no counterpart"
# D204 -> Q207
# R208 -> N211
# V210 -> D213
# Y211 -> P214
# L212 -> gap
# N348 -> K350
# K371 -> Q373

domain_aligner = Align.PairwiseAligner()
domain_aligner.mode = "global"
domain_aligner.substitution_matrix = substitution_matrices.load("BLOSUM62")
domain_aligner.open_gap_score = -11
domain_aligner.extend_gap_score = -1

human_apical = human_seq[81:91] + human_seq[92:95] + human_seq[227] + human_seq[250]
mouse_apical = mouse_seq[82:96] + mouse_seq[227] + mouse_seq[250]

domain_alignments = domain_aligner.align(human_apical, mouse_apical)
domain_alignment = domain_alignments[0]

print(f"score:      {domain_alignment.score:.1f}")
print(f"alignments: {len(domain_alignment)}")

print("Apical Domains\n")
print(domain_alignment)

score:      20.0
alignments: 2
Apical Domains

target            0 IVDKNGRLVYVENNK- 15
                  0 ||..||.|..||..|- 16
query             0 IVQSNGNLDPVESPKQ 16

